# Entrenamiento y Evaluación de Mendicant Bias
Este notebook ejecuta el pipeline de entrenamiento Maestro-Aprendiz usando ConvNeXtV2 Atto y el Oráculo congelado Daowa-maad.

In [ ]:
import torch
import sys
import os
import matplotlib.pyplot as plt

sys.path.append('utils/train')
sys.path.append('utils/models')

from mendicant_dataset import get_mendicant_dataloaders
from trainer_mendicant import MendicantTrainer
from mendicant_bias import MendicantBias_ConvNeXt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

## 1. Cargar Datos y Modelos

In [ ]:
# Rutas de datos (Ajusta los CSVs según tu split de train/val)
csv_train = "labels/dataset_mendicantV3.csv" 
csv_val = "labels/dataset_mendicantV3.csv" # TODO: Reemplazar con el CSV real de val
img_dir = "data/PetImages"

loaders = get_mendicant_dataloaders(csv_train, csv_val, img_dir, batch_size=32, num_workers=4)
print("DataLoaders listos.")

# Cargar Oráculo y Aprendiz
oracle = torch.jit.load('weights/Daowa_Oracle_Frozen.pt', map_location=device)
mendicant = MendicantBias_ConvNeXt(pretrained=True)
print("Modelos instanciados.")

## 2. Iniciar Entrenamiento

In [ ]:
# Inicializar el Trainer
trainer = MendicantTrainer(mendicant, oracle, loaders, device, save_dir="Models")

# Iniciar entrenamiento (Esto tomara tiempo)
# Opcional: puedes cambiar trainer.epochs = 10 temporalmente si quieres hacer pruebas cortas.
trainer.train(history_file="logs/history_mendicant.csv")

## 3. Evaluar Métricas Globales
Al terminar el entrenamiento, ejecuta esta celda para generar la Matriz de Confusión, la Curva ROC y las gráficas de Aprendizaje.

In [ ]:
from eval_mendicant import plot_training_curves, generate_classification_metrics

# Plotear curvas de perdida y accuracy
plot_training_curves("logs/history_mendicant.csv", save_dir="logs/evaluations")

# Evaluar sobre el validation loader para sacar Matriz de Confusion y ROC
generate_classification_metrics(mendicant, oracle, loaders["val"], device, save_dir="logs/evaluations")

## 4. Grad-CAM (Explicabilidad Visual)
Analiza en qué parte de la imagen/máscara se fijó ConvNeXtV2 para tomar su decisión final.

In [ ]:
from gradcam_mendicant import generate_gradcam_mendicant

# Elegir una imagen de validacion
test_img = "data/PetImages/Cat/0.jpg"  # Cambiar por la ruta de la imagen que desees auditar
clase_objetivo = 0  # 0 = Gato, 1 = Perro

generate_gradcam_mendicant(
    mendicant_model=mendicant, 
    oracle_model=oracle, 
    img_path=test_img, 
    target_class=clase_objetivo, 
    device=device,
    save_dir="logs/evaluations"
)